In [37]:
import sys
sys.path.append('..')

In [38]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob

In [39]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [40]:
# Paths
IXI_BASE = "../data/raw/ixi/IXI_data/IXI_data"
IXI_CSV = "../data/raw/ixi/IXI_processed.csv"

In [41]:
meta_df = pd.read_csv(IXI_CSV)
print(f"Total subjects in CSV: {len(meta_df)}")
meta_df.head()

Total subjects in CSV: 587


,Subject_ID,"SEX_ID (1=m, 2=f)",HEIGHT,WEIGHT,ETHNIC_ID,MARITAL_ID,OCCUPATION_ID,QUALIFICATION_ID,DOB,DATE_AVAILABLE,STUDY_DATE,AGE
0,0,2,164,58,1,4,1,5,1/30/1970,1,11/18/2005,35.800137
1,1,1,175,70,1,2,1,5,8/20/1966,1,6/1/2005,38.781656
2,2,1,182,70,1,2,1,5,9/15/1958,1,6/1/2005,46.710472
3,3,2,163,65,1,4,1,5,3/15/1971,1,6/9/2005,34.236824
4,4,1,181,90,2,1,6,5,3/11/1981,1,6/23/2005,24.284736


In [42]:
meta_df.rename(columns={'Subject_ID': 'subject_id', 'SEX_ID (1=m, 2=f)': 'sex_code', 'AGE': 'age'}, inplace=True)
meta_df['sex'] = meta_df['sex_code'].map({1: 'Male', 2: 'Female'})


In [43]:
meta_df.columns

Index(['subject_id', 'sex_code', 'HEIGHT', 'WEIGHT', 'ETHNIC_ID', 'MARITAL_ID',
       'OCCUPATION_ID', 'QUALIFICATION_ID', 'DOB', 'DATE_AVAILABLE',
       'STUDY_DATE', 'age', 'sex'],
      dtype='object')

In [44]:
meta_df.head()

,subject_id,sex_code,HEIGHT,WEIGHT,ETHNIC_ID,MARITAL_ID,OCCUPATION_ID,QUALIFICATION_ID,DOB,DATE_AVAILABLE,STUDY_DATE,age,sex
0,0,2,164,58,1,4,1,5,1/30/1970,1,11/18/2005,35.800137,Female
1,1,1,175,70,1,2,1,5,8/20/1966,1,6/1/2005,38.781656,Male
2,2,1,182,70,1,2,1,5,9/15/1958,1,6/1/2005,46.710472,Male
3,3,2,163,65,1,4,1,5,3/15/1971,1,6/9/2005,34.236824,Female
4,4,1,181,90,2,1,6,5,3/11/1981,1,6/23/2005,24.284736,Male


In [45]:
def get_split_subjects(split_dir):
    pkl_files = glob(os.path.join(IXI_BASE, split_dir, "*.pkl"))
    subjects = []
    for f in pkl_files:
        # Extract number from "subject_327.pkl"
        fname = os.path.basename(f)
        num_str = fname.split('_')[-1].replace('.pkl', '')
        subjects.append(int(num_str))
    return subjects

In [46]:
train_subjects = get_split_subjects("Train")
val_subjects = get_split_subjects("Val")
test_subjects = get_split_subjects("Test")

In [47]:
print(f"Train: {len(train_subjects)}, Val: {len(val_subjects)}, Test: {len(test_subjects)}")

Train: 403, Val: 58, Test: 115


In [48]:
# Add split column
meta_df['split'] = 'unknown'
meta_df.loc[meta_df['subject_id'].isin(train_subjects), 'split'] = 'train'
meta_df.loc[meta_df['subject_id'].isin(val_subjects), 'split'] = 'val'
meta_df.loc[meta_df['subject_id'].isin(test_subjects), 'split'] = 'test'


In [49]:
# Check for missing
missing = meta_df[meta_df['split'] == 'unknown']
print(f"Subjects in CSV but not in any split folder: {len(missing)}")
meta_df['split'].value_counts()

Subjects in CSV but not in any split folder: 1


split
train      412
test       113
val         61
unknown      1
Name: count, dtype: int64

In [50]:
plot_age_histogram(meta_df['AGE'], title="IXI: Age Distribution (all splits)", save_path="../analysis/figures/ixi_age_all.png")

NameError: name 'plot_age_histogram' is not defined